In [1]:
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

In [2]:
from FlagEmbedding import BGEM3FlagModel

d:\Github_ThisPC\muict_thai_legal_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


In [3]:
from pathlib import Path

# --- 1. Path & Connection Configuration ---
BASE_DIR = Path.cwd().parent
DATASET_PATH = BASE_DIR / "data" / "processed" / "nitibench_cleaned_2026-03-17.parquet"

# เปลี่ยนจาก path เป็น url ของ localhost
QDRANT_URL = "http://localhost:6333" 
COLLECTION_NAME = "thai_laws_collection"
MODEL_NAME = "VISAI-AI/nitibench-ccl-human-finetuned-bge-m3"

In [5]:
import torch
from FlagEmbedding import BGEM3FlagModel

# --- 2. Initialize Client & Model ---
client = QdrantClient(url=QDRANT_URL) # เชื่อมต่อผ่าน URL

# เช็คว่ามี GPU (CUDA) หรือไม่
# ถ้ามีให้ใช้ 'cuda' ถ้าไม่มีให้ใช้ 'cpu'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_fp16 = True if device == 'cuda' else False  # fp16 ส่วนใหญ่รองรับเฉพาะบน GPU

print(f"กำลังรันโมเดลบน: {device.upper()}")

# 2. Initialize Model
model = BGEM3FlagModel(
    'VISAI-AI/nitibench-ccl-human-finetuned-bge-m3', 
    use_fp16=use_fp16, 
    device=device 
)

กำลังรันโมเดลบน: CUDA


Fetching 23 files: 100%|██████████| 23/23 [00:00<?, ?it/s]


In [6]:
from FlagEmbedding import BGEM3FlagModel
import torch
# วิธีตรวจสอบ
print(f"Device being used: {model.device}")

# อีกวิธีคือเช็คผ่าน torch โดยตรง (ถ้า model ใช้ torch เป็น backend)
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"Current GPU device name: {torch.cuda.get_device_name(0)}")

Device being used: cuda
Is CUDA available? True
Current GPU device name: NVIDIA GeForce GTX 1650


In [ ]:
# --- 3. Create Collection ---
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
    )
    print(f"สร้าง Collection: {COLLECTION_NAME} บน Server เรียบร้อย")

In [7]:
# --- 4. Process & Ingest ---
df = pd.read_parquet(DATASET_PATH)
total_rows = len(df)
batch_size = 50

# --- ขั้นตอนที่ 1: เตรียมข้อมูลทั้งหมดให้เป็น List ของ PointStruct ---
# (ขั้นตอนนี้จะใช้เวลานานที่สุดเพราะต้องทำ Embedding)
print(f"1. เริ่มการสร้าง Embedding สำหรับข้อมูลทั้งหมด {total_rows} รายการ...")

all_points = []
for idx, row in df.iterrows():
    text_to_embed = row['section_content']
    
    # แปลงเป็น Vector (BGE-M3)
    output = model.encode([text_to_embed], return_dense=True)
    vector = output['dense_vecs'][0].tolist()
    
    # เตรียมโครงสร้างข้อมูล
    payload = {
        "text": text_to_embed,
        "law_name": str(row['law_name']),
        "section_num": str(row['section_num'])
    }
    
    all_points.append(PointStruct(id=idx, vector=vector, payload=payload))
    
    # Print Progress เฉพาะขั้นตอนการทำ Embedding
    if (idx + 1) % 100 == 0 or (idx + 1) == total_rows:
        print(f"Embedding Progress: [{idx + 1}/{total_rows}] รายการเรียบร้อยแล้ว")

1. เริ่มการสร้าง Embedding สำหรับข้อมูลทั้งหมด 3833 รายการ...


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Embedding Progress: [100/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [200/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [300/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [400/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [500/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [600/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [700/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [800/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [900/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [1000/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [1100/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [1200/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [1300/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [1400/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [1500/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [1600/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [1700/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [1800/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [1900/3833] รายการเรียบร้อยแล้ว
Embedding Progress: [

In [8]:
import pandas as pd

# --- ขั้นตอนที่ 2: แปลง all_points เป็น DataFrame เพื่อ Backup ---
print(f"2. กำลังแปลงข้อมูล {len(all_points)} รายการ เป็น DataFrame...")

# ดึงข้อมูลจาก PointStruct ออกมาทีละส่วน
backup_data = []
for point in all_points:
    backup_data.append({
        "id": point.id,
        "law_name": point.payload.get("law_name"),
        "text": point.payload.get("text"),
        "section_num": point.payload.get("section_num"),
        "vector": point.vector  # เก็บเป็น List of Floats ตรงๆ ได้เลย
    })

# สร้าง DataFrame
df_backup = pd.DataFrame(backup_data)

OUTPUT_DIR = BASE_DIR / "data" / "processed" 
# บันทึกเป็น Parquet
BACKUP_FILE_PATH = OUTPUT_DIR / "embeddings_nitibench-ccl-human-finetuned-bge-m3_backup.parquet"
df_backup.to_parquet(BACKUP_FILE_PATH, index=False, engine='pyarrow')

print(f"✅ บันทึก Backup เรียบร้อยแล้วที่: {BACKUP_FILE_PATH}")
print(f"ขนาดของไฟล์: {df_backup.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

2. กำลังแปลงข้อมูล 3833 รายการ เป็น DataFrame...
✅ บันทึก Backup เรียบร้อยแล้วที่: d:\Github_ThisPC\muict_thai_legal_rag\data\processed\embeddings_nitibench-ccl-human-finetuned-bge-m3_backup.parquet
ขนาดของไฟล์: 35.33 MB


In [ ]:
# --- ขั้นตอนที่ 2: เริ่มส่งข้อมูลเข้า Qdrant (Ingestion) ---
# มั่นใจได้ว่าข้อมูลทุกอย่างพร้อมแล้วใน all_points
print(f"2. เริ่มส่งข้อมูลเข้า Qdrant (Batch size: {batch_size})...")

for i in range(0, len(all_points), batch_size):
    # ดึงข้อมูลออกมาทีละ Batch (0-50, 50-100, ...)
    batch = all_points[i : i + batch_size]
    
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch
    )
    
    # คำนวณความคืบหน้าการส่ง
    current_count = min(i + batch_size, total_rows)
    percentage = (current_count / total_rows) * 100
    print(f"Ingestion Progress: [{current_count}/{total_rows}] ({percentage:.2f}%)")

print(f"\n[สำเร็จ] ข้อมูลทั้งหมด {total_rows} รายการ ถูกบันทึกลงใน Qdrant เรียบร้อยแล้ว!")